In [ ]:
#| hide
from nbdev import show_doc

# Pairwise Relation Geometry

## Diagnostics via Projection into a Low-Dimensional Space

In [ ]:
#| eval: false
from fhemb.utils.cutils import plot_pr_projection

In [ ]:
#| eval: false

show_doc(plot_pr_projection, title_level=4, name="Plot pairwise relation projection")

---

#### Plot pairwise relation projection

```python

def plot_pr_projection(
    groups:VAR_POSITIONAL, # Each group is an array of shape (n_samples, n_features, n_timesteps).
Groups are preserved for coloring and labeling.
    alignment_type:str='fastdtw', # 'dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div'.
    gamma:float=1.0, # Soft-DTW smoothing parameter.
    dist:function=euclidean, # Alignment metric function for fastdtw.
    p:int=3, # Minkowski parameter (if used).
    radius:int=1, # FastDTW radius.
    projection:str='mds', # 'mds', 'pca', 'tsne', 'umap'.
    transformation:str='gauss', # Distance→similarity transform for PCA: 'gauss', 'invert', 'minmax'.
    normalize:NoneType=None, # Similarity normalization for distance2similarity: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'none'.
    n_jobs:int=4, # Parallel workers for pairwise relation computation.
    labels:NoneType=None, # Optional labels for each time series (flattened order).
    title:str='Pairwise Relation Matrix Projection', # Plot title.
    dimensions:int=2, # 2 or 3.
):


```

*Canonical visualization of pairwise relation matrices using MDS, PCA, TSNE or UMAP.*

::: {.callout collapse="true" title= "Similarity Transformations for PCA - Summary Table"}
| `transformation` value | Formula | Intuition | Strengths | Notes |
|--------|---------|-----------|-----------|--------|
| **Gaussian (RBF) Similarity** | $K_{ij} = \exp\!\left(-\frac{D_{ij}^2}{2\sigma^2}\right)$ | Nearby points get high similarity; far points decay smoothly | Handles nonlinear structure; tunable locality via $\sigma$ | Most common kernel for PCA on distance‑based data |
| **Inverse Distance Similarity** | $K_{ij} = \frac{1}{1 + D_{ij}}$ | Similarity decreases monotonically with distance | Simple, bounded, interpretable | Good when distances vary widely |
| **Min‑Max Normalized Similarity** | $K_{ij} = 1 - \frac{D_{ij} - D_{\min}}{D_{\max} - D_{\min}}$ | Linearly maps distances to $[0,1]$ | Preserves rank; easy to visualize | Useful when PCA expects normalized inputs |
:::

::: {.callout-caution title="transformation value"}
Applies to PCA only.
::: 

## Diagnostics via Singular Value Decomposition (SVD)

In [ ]:
#| eval: false
from fhemb.utils.cutils import svd_pr

In [ ]:
#| eval: false

show_doc(svd_pr, title_level=4)

---

#### svd_pr

```python

def svd_pr(
    ts, # Arrays of shape (n_samples, n_timesteps) or (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use ('dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # smoothing parameter (for soft_dtw only):
    Small gamma → behaves more like classic DTW (hard minimum)
    Large gamma → smoother, more diffused alignment (less sensitive to exact path)
    0.01 Very close to DTW:     Precise alignment
    1.0 Balanced smoothing:     Most common default
    10.0 Very smooth: Robust to noise, good for optimization
    dist:function=euclidean, # Alignment metric function to use for DTW (only for fastdtw).
    p:int=3, # Parameter for Minkowski alignment metric (if used).
    radius:int=1, # Radius parameter for the FastDTW algorithm.
    transformation:str='gauss', # Method to transform distances into similarities.
    Options: 'gauss', 'invert', 'minmax'
    normalize:NoneType=None, # Normalization method to apply to the similarity matrix.
    Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'none'
    n_jobs:int=4, # number of jobs for parallel execution
): # U: Left singular vectors
S: Singular values (1D array)
Vt: Transpose of right singular vectors


```

*Compute the Singular Value Decomposition (SVD) of a pairwise relation derived from time series distances.*
The pairwise relation matrix is calculated using distance correlation (dcor) or a DTW method.

::: {.callout-note collapse="false" title="svd_pr"}
Computes a **low-rank decomposition** of a distance-based similarity matrix using **SVD**. This function helps analyze the structure of the similarity geometry and estimate the **intrinsic dimensionality** of the dataset.

**Pipeline**

1. Compute pairwise distances using `alignment_type` (e.g., `fastdtw`).  
2. Convert distances into similarities using `transformation` (e.g., `gauss`, `invert`, `minmax`).  
3. Optionally normalize time series using `normalize`.  
4. Apply SVD to the resulting similarity matrix $\qquad K = U \Sigma V^\top$

**Returns**

- `U` — left singular vectors  
- `S` — singular values  
- `Vt` — right singular vectors  

**Interpretation**

- Singular values `S` quantify how much structure each latent dimension explains.  
- Their decay provides a practical estimate of the **intrinsic dimension** of the similarity space:  
  - sharp drop → low intrinsic dimension  
  - gradual decay → diffuse or noisy structure  
- Conceptually similar to PCA, but applied directly to the **similarity matrix** rather than centered feature data.

**Use cases**

- Inspecting geometric structure before embedding  
- Choosing the number of PCA/MDS components  
- Detecting noise vs. signal in distance‑based representations  
- Low‑rank smoothing or compression of similarity matrices
:::